# Analisis de linea usando YOLO ROI + Sobel Y

Objetivo: usar YOLO solo para ubicar la ROI del paquete/tubo y, dentro de esa ROI, analizar **todos los edges** detectados con Sobel Y para construir una linea robusta.

Esta notebook compara tres ideas:

1. **Sobel Y + mascara de edges**: ver si el edge que necesitamos aparece realmente dentro de la caja.
2. **RANSAC sobre todos los edge pixels**: no depende de dos puntos ni de un maximo por columna.
3. **Refinamiento por proyeccion en X**: despues de una linea base, agrupa edge pixels por columnas y ajusta una linea usando la evidencia cercana.

La intencion es iterar visualmente antes de meter otra version a la app.

In [ ]:
from pathlib import Path
import json
import math
import random

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

ROOT = Path.cwd()
VIDEO_PATH = Path(r"C:\Users\luis_\Downloads\20260508_000307_7F66.mkv")
HOMOGRAPHY_PATH = ROOT / "outputs" / "homography_selection.json"
MODEL_PATH = ROOT / "runs" / "detect" / "runs_tx2" / "yolo11n_tubos_v1" / "weights" / "best.pt"
OUT_DIR = ROOT / "outputs" / "sobel_edge_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Frames/segundos interesantes. Cambia esta lista para analizar otros casos.
PREVIEW_SECONDS = [84, 85, 100, 112, 115, 118, 120]

# Umbrales iniciales. La idea es moverlos y observar.
YOLO_CONF = 0.10
IMG_SIZE = 960
ROI_PAD_X = 0
ROI_PAD_Y = 0
EDGE_PERCENTILE = 86
EDGE_BAND = (0.0, 1.0)  # fraccion vertical dentro de ROI: (0.0, 1.0) usa toda la box
MAX_ABS_SLOPE = 0.45
RANSAC_TOL = 5.0
RANSAC_ITERS = 1200
PROJECTION_BIN_WIDTH = 6
PROJECTION_RADIUS = 14

print('video:', VIDEO_PATH)
print('homography:', HOMOGRAPHY_PATH)
print('model:', MODEL_PATH)
print('out:', OUT_DIR)

In [ ]:
def load_homography(path):
    payload = json.loads(path.read_text(encoding='utf-8'))
    H = np.asarray(payload['homography_matrix'], dtype=np.float64)
    out_size = tuple(int(v) for v in payload['output_size'])
    return H, out_size, payload

H, WARP_SIZE, HOMOGRAPHY_DATA = load_homography(HOMOGRAPHY_PATH)
model = YOLO(str(MODEL_PATH))

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f'No se pudo abrir video: {VIDEO_PATH}')
FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
DURATION = TOTAL_FRAMES / FPS
cap.release()

print(f'fps={FPS:.3f} frames={TOTAL_FRAMES} duration={DURATION:.2f}s warp_size={WARP_SIZE}')

In [ ]:
def read_frame_at_second(second):
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    frame_idx = max(0, min(int(round(second * FPS)), TOTAL_FRAMES - 1))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f'No se pudo leer frame {frame_idx} @ {second}s')
    return frame_idx, frame


def rectify(frame):
    return cv2.warpPerspective(frame, H, WARP_SIZE)


def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def predict_boxes(rectified, conf=YOLO_CONF, imgsz=IMG_SIZE):
    result = model.predict(rectified, conf=conf, imgsz=imgsz, verbose=False)[0]
    if result.boxes is None or len(result.boxes) == 0:
        return []
    xyxy = result.boxes.xyxy.detach().cpu().numpy()
    scores = result.boxes.conf.detach().cpu().numpy()
    boxes = []
    for box, score in zip(xyxy, scores):
        x0, y0, x1, y1 = [float(v) for v in box.tolist()]
        boxes.append({
            'x': x0,
            'y': y0,
            'w': max(0.0, x1 - x0),
            'h': max(0.0, y1 - y0),
            'conf': float(score),
        })
    return sorted(boxes, key=lambda b: b['w'] * b['h'] * b['conf'], reverse=True)


def best_box(rectified, boxes):
    if not boxes:
        return None
    return max(boxes, key=lambda b: b['w'] * b['h'] * b.get('conf', 1.0))


def clamp_box(box, image_shape, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y):
    h, w = image_shape[:2]
    x0 = int(np.floor(box['x'] - pad_x))
    y0 = int(np.floor(box['y'] - pad_y))
    x1 = int(np.ceil(box['x'] + box['w'] + pad_x))
    y1 = int(np.ceil(box['y'] + box['h'] + pad_y))
    x0 = max(0, min(x0, w - 2))
    y0 = max(0, min(y0, h - 2))
    x1 = max(x0 + 2, min(x1, w - 1))
    y1 = max(y0 + 2, min(y1, h - 1))
    return x0, y0, x1, y1


def crop_roi(rectified, box, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y):
    x0, y0, x1, y1 = clamp_box(box, rectified.shape, pad_x, pad_y)
    return rectified[y0:y1, x0:x1].copy(), (x0, y0, x1, y1)

In [ ]:
def odd(v):
    v = int(v)
    return v if v % 2 == 1 else v + 1


def build_sobel_y_edges(roi, clahe_clip=2.0, blur_ksize=(31, 1), sobel_ksize=3):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8)).apply(gray)
    blur = cv2.GaussianBlur(clahe, blur_ksize, 0)
    sobel_y = cv2.Sobel(blur, cv2.CV_32F, 0, 1, ksize=sobel_ksize)
    edge = np.abs(sobel_y)
    return gray, clahe, blur, edge


def make_edge_mask(edge, percentile=EDGE_PERCENTILE, band=EDGE_BAND):
    h, w = edge.shape
    y0 = int(round(band[0] * h))
    y1 = int(round(band[1] * h))
    y0 = max(0, min(y0, h - 2))
    y1 = max(y0 + 2, min(y1, h))
    band_edge = edge[y0:y1]
    threshold = np.percentile(band_edge, percentile)
    mask = np.zeros_like(edge, dtype=np.uint8)
    mask[y0:y1] = (edge[y0:y1] >= threshold).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
    return mask, threshold, (y0, y1)


def edge_points(edge, mask, max_points=12000):
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return np.empty((0, 2), dtype=np.float32), np.empty((0,), dtype=np.float32)
    weights = edge[ys, xs].astype(np.float32)
    pts = np.column_stack([xs, ys]).astype(np.float32)
    if len(pts) > max_points:
        # Muestreo ponderado para mantener edges fuertes sin matar variedad espacial.
        p = weights / (weights.sum() + 1e-9)
        idx = np.random.default_rng(42).choice(len(pts), size=max_points, replace=False, p=p)
        pts = pts[idx]
        weights = weights[idx]
    return pts, weights


def weighted_line_fit(points, weights=None):
    x = points[:, 0].astype(np.float64)
    y = points[:, 1].astype(np.float64)
    if weights is None:
        weights = np.ones_like(x)
    else:
        weights = weights.astype(np.float64)
    A = np.column_stack([x, np.ones_like(x)])
    W = np.sqrt(weights + 1e-9)[:, None]
    coeff, *_ = np.linalg.lstsq(A * W, y * W[:, 0], rcond=None)
    slope, intercept = coeff
    return float(slope), float(intercept)


def line_residual(points, slope, intercept):
    x = points[:, 0]
    y = points[:, 1]
    return np.abs(slope * x - y + intercept) / np.sqrt(slope * slope + 1.0)


def ransac_line(points, weights=None, iterations=RANSAC_ITERS, tol=RANSAC_TOL, max_abs_slope=MAX_ABS_SLOPE, seed=42):
    if len(points) < 2:
        return None
    rng = np.random.default_rng(seed)
    if weights is None:
        weights = np.ones(len(points), dtype=np.float32)
    p = weights / (weights.sum() + 1e-9)
    best = None
    idx_all = np.arange(len(points))
    for _ in range(iterations):
        i, j = rng.choice(idx_all, size=2, replace=False, p=p)
        x1, y1 = points[i]
        x2, y2 = points[j]
        dx = float(x2 - x1)
        if abs(dx) < 8:
            continue
        slope = float((y2 - y1) / dx)
        if abs(slope) > max_abs_slope:
            continue
        intercept = float(y1 - slope * x1)
        residual = line_residual(points, slope, intercept)
        inliers = residual <= tol
        if inliers.sum() < 8:
            continue
        score = float(weights[inliers].sum()) * float(inliers.sum())
        if best is None or score > best['score']:
            best = {'slope': slope, 'intercept': intercept, 'inliers': inliers, 'score': score}
    if best is None:
        return None
    slope, intercept = weighted_line_fit(points[best['inliers']], weights[best['inliers']])
    residual = line_residual(points, slope, intercept)
    inliers = residual <= tol
    crm = float(np.sqrt(np.mean(residual[inliers] ** 2))) if inliers.any() else float('inf')
    confidence = float(inliers.sum()) / float(len(points))
    confidence *= float(np.median(weights[inliers])) / (float(weights.mean()) + 1e-9) if inliers.any() else 0.0
    return {
        'slope': slope,
        'intercept': intercept,
        'inliers': inliers,
        'crm_px': crm,
        'confidence': confidence,
        'num_points': int(len(points)),
        'num_inliers': int(inliers.sum()),
    }


def refine_by_x_projection(points, weights, base_line, bin_width=PROJECTION_BIN_WIDTH, radius=PROJECTION_RADIUS):
    if base_line is None or len(points) == 0:
        return None
    slope = base_line['slope']
    intercept = base_line['intercept']
    xs = points[:, 0]
    ys = points[:, 1]
    residual = np.abs(ys - (slope * xs + intercept))
    near = residual <= radius
    if near.sum() < 8:
        return None
    pts = points[near]
    w = weights[near]
    min_x = int(np.floor(pts[:, 0].min()))
    max_x = int(np.ceil(pts[:, 0].max()))
    proj_points = []
    proj_weights = []
    for x0 in range(min_x, max_x + 1, bin_width):
        x1 = x0 + bin_width
        m = (pts[:, 0] >= x0) & (pts[:, 0] < x1)
        if m.sum() < 2:
            continue
        yy = pts[m, 1]
        ww = w[m]
        # Centroide ponderado de todos los edge pixels cercanos a la linea base.
        y_hat = float(np.sum(yy * ww) / (np.sum(ww) + 1e-9))
        x_hat = float((x0 + x1) / 2)
        proj_points.append([x_hat, y_hat])
        proj_weights.append(float(np.sum(ww)))
    if len(proj_points) < 4:
        return None
    proj_points = np.asarray(proj_points, dtype=np.float32)
    proj_weights = np.asarray(proj_weights, dtype=np.float32)
    slope2, intercept2 = weighted_line_fit(proj_points, proj_weights)
    residual2 = line_residual(proj_points, slope2, intercept2)
    return {
        'slope': slope2,
        'intercept': intercept2,
        'points': proj_points,
        'weights': proj_weights,
        'crm_px': float(np.sqrt(np.mean(residual2 ** 2))),
        'num_points': int(len(proj_points)),
    }


def hough_lines(mask, max_abs_slope=MAX_ABS_SLOPE):
    img = (mask * 255).astype(np.uint8)
    lines = cv2.HoughLinesP(img, rho=1, theta=np.pi/180, threshold=30, minLineLength=80, maxLineGap=18)
    if lines is None:
        return []
    out = []
    for line in lines[:, 0, :]:
        x1, y1, x2, y2 = [float(v) for v in line]
        if abs(x2 - x1) < 1:
            continue
        slope = (y2 - y1) / (x2 - x1)
        if abs(slope) > max_abs_slope:
            continue
        intercept = y1 - slope * x1
        length = math.hypot(x2 - x1, y2 - y1)
        out.append({'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'slope': slope, 'intercept': intercept, 'length': length})
    return sorted(out, key=lambda l: l['length'], reverse=True)

In [ ]:
def analyze_second(second, conf=YOLO_CONF, edge_percentile=EDGE_PERCENTILE, band=EDGE_BAND, show=True):
    frame_idx, raw = read_frame_at_second(second)
    rectified = rectify(raw)
    boxes = predict_boxes(rectified, conf=conf)
    box = best_box(rectified, boxes)
    if box is None:
        print(f'{second:.2f}s frame={frame_idx}: YOLO no detecto ROI')
        return None

    roi, roi_px = crop_roi(rectified, box, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y)
    gray, clahe, blur, edge = build_sobel_y_edges(roi)
    mask, threshold, band_px = make_edge_mask(edge, percentile=edge_percentile, band=band)
    pts, weights = edge_points(edge, mask)
    ransac = ransac_line(pts, weights)
    refined = refine_by_x_projection(pts, weights, ransac)
    hough = hough_lines(mask)

    result = {
        'second': second,
        'frame_idx': frame_idx,
        'rectified': rectified,
        'box': box,
        'roi': roi,
        'roi_px': roi_px,
        'gray': gray,
        'clahe': clahe,
        'edge': edge,
        'mask': mask,
        'threshold': threshold,
        'band_px': band_px,
        'points': pts,
        'weights': weights,
        'ransac': ransac,
        'refined': refined,
        'hough': hough,
    }
    if show:
        plot_analysis(result)
    return result


def draw_line_on_axis(ax, line, width, color='lime', label='line'):
    if line is None:
        return
    if 'x1' in line:
        ax.plot([line['x1'], line['x2']], [line['y1'], line['y2']], color=color, lw=2, label=label)
        return
    xs = np.array([0, width - 1], dtype=np.float32)
    ys = line['slope'] * xs + line['intercept']
    ax.plot(xs, ys, color=color, lw=2, label=label)


def plot_analysis(result):
    roi = result['roi']
    edge = result['edge']
    mask = result['mask']
    pts = result['points']
    weights = result['weights']
    ransac = result['ransac']
    refined = result['refined']
    hough = result['hough']
    box = result['box']
    x0, y0, x1, y1 = result['roi_px']

    fig, axes = plt.subplots(2, 3, figsize=(18, 9), constrained_layout=True)
    axes = axes.ravel()

    rect_vis = result['rectified'].copy()
    cv2.rectangle(rect_vis, (x0, y0), (x1, y1), (0, 180, 255), 2)
    axes[0].imshow(bgr_to_rgb(rect_vis))
    axes[0].set_title(f"Frame {result['frame_idx']} @ {result['second']:.2f}s | YOLO {box.get('conf', 1):.2f}")
    axes[0].axis('off')

    axes[1].imshow(bgr_to_rgb(roi))
    axes[1].set_title('ROI YOLO estricta')
    axes[1].axis('off')

    axes[2].imshow(edge, cmap='magma')
    axes[2].set_title(f"Sobel Y abs | thr p{EDGE_PERCENTILE}={result['threshold']:.1f}")
    axes[2].axis('off')

    axes[3].imshow(mask, cmap='gray')
    axes[3].set_title(f"Mascara edges | puntos={len(pts)}")
    axes[3].axis('off')

    axes[4].imshow(bgr_to_rgb(roi))
    if len(pts):
        inliers = ransac['inliers'] if ransac is not None else np.zeros(len(pts), dtype=bool)
        axes[4].scatter(pts[~inliers, 0], pts[~inliers, 1], s=2, c='orange', alpha=0.35, label='edges')
        axes[4].scatter(pts[inliers, 0], pts[inliers, 1], s=4, c='lime', alpha=0.65, label='inliers')
    if ransac is not None:
        draw_line_on_axis(axes[4], ransac, roi.shape[1], color='cyan', label='RANSAC')
    if refined is not None:
        axes[4].scatter(refined['points'][:, 0], refined['points'][:, 1], s=18, c='white', marker='x', label='proj X')
        draw_line_on_axis(axes[4], refined, roi.shape[1], color='lime', label='refined')
    if hough:
        draw_line_on_axis(axes[4], hough[0], roi.shape[1], color='red', label='Hough')
    axes[4].set_title('Todos los edge pixels + lineas candidatas')
    axes[4].legend(loc='lower right')
    axes[4].set_xlim(0, roi.shape[1])
    axes[4].set_ylim(roi.shape[0], 0)

    overlay = result['rectified'].copy()
    cv2.rectangle(overlay, (x0, y0), (x1, y1), (0, 180, 255), 2)
    if refined is not None:
        xs = np.array([0, roi.shape[1] - 1], dtype=np.float32)
        ys = refined['slope'] * xs + refined['intercept']
        cv2.line(overlay, (x0 + int(xs[0]), y0 + int(ys[0])), (x0 + int(xs[1]), y0 + int(ys[1])), (0, 255, 0), 3)
    elif ransac is not None:
        xs = np.array([0, roi.shape[1] - 1], dtype=np.float32)
        ys = ransac['slope'] * xs + ransac['intercept']
        cv2.line(overlay, (x0 + int(xs[0]), y0 + int(ys[0])), (x0 + int(xs[1]), y0 + int(ys[1])), (255, 255, 0), 3)
    axes[5].imshow(bgr_to_rgb(overlay))
    axes[5].set_title('Linea proyectada en frame rectificado')
    axes[5].axis('off')

    if ransac is not None:
        print('RANSAC:', {k: round(v, 4) if isinstance(v, float) else v for k, v in ransac.items() if k != 'inliers'})
    if refined is not None:
        print('Refined X projection:', {k: round(v, 4) if isinstance(v, float) else v for k, v in refined.items() if k not in ('points', 'weights')})
    if hough:
        print('Best Hough:', {k: round(v, 4) if isinstance(v, float) else v for k, v in hough[0].items()})
    plt.show()

## Analisis de un frame

Empieza con un frame donde sabemos que YOLO detecta algo. Si la linea cae mal, cambia:

- `EDGE_PERCENTILE`: mas bajo mete mas edges; mas alto deja solo edges fuertes.
- `EDGE_BAND`: limita verticalmente la busqueda dentro de la caja, por ejemplo `(0.45, 1.0)` para la mitad inferior.
- `RANSAC_TOL` y `PROJECTION_RADIUS`: controlan que tan lejos de la linea base aceptamos edges.

In [ ]:
result = analyze_second(115.0, conf=0.10, edge_percentile=86, band=(0.0, 1.0), show=True)

## Barrido visual por varios segundos

Este bloque genera una cuadricula rapida. Sirve para responder: "?la linea que buscamos aparece de forma consistente con todos los edges?".

In [ ]:
def summarize_result(result):
    if result is None:
        return {'ok': False}
    r = result['ransac']
    p = result['refined']
    return {
        'second': result['second'],
        'frame_idx': result['frame_idx'],
        'yolo_conf': result['box'].get('conf', 1.0),
        'edge_points': len(result['points']),
        'ransac_conf': None if r is None else r['confidence'],
        'ransac_crm': None if r is None else r['crm_px'],
        'proj_crm': None if p is None else p['crm_px'],
        'line': 'projection' if p is not None else ('ransac' if r is not None else 'none'),
    }

summaries = []
for sec in PREVIEW_SECONDS:
    res = analyze_second(sec, conf=0.10, edge_percentile=86, band=(0.0, 1.0), show=False)
    summaries.append(summarize_result(res))

summaries

In [ ]:
def plot_grid(seconds, conf=0.10, edge_percentile=86, band=(0.0, 1.0)):
    n = len(seconds)
    cols = 2
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4.8 * rows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, sec in zip(axes, seconds):
        res = analyze_second(sec, conf=conf, edge_percentile=edge_percentile, band=band, show=False)
        if res is None:
            ax.text(0.5, 0.5, f'{sec:.1f}s\nSin ROI YOLO', ha='center', va='center')
            ax.axis('off')
            continue
        overlay = res['rectified'].copy()
        x0, y0, x1, y1 = res['roi_px']
        cv2.rectangle(overlay, (x0, y0), (x1, y1), (0, 180, 255), 2)
        line_src = res['refined'] or res['ransac']
        if line_src is not None:
            xs = np.array([0, res['roi'].shape[1] - 1], dtype=np.float32)
            ys = line_src['slope'] * xs + line_src['intercept']
            cv2.line(overlay, (x0 + int(xs[0]), y0 + int(ys[0])), (x0 + int(xs[1]), y0 + int(ys[1])), (0, 255, 0), 3)
        title = f"{sec:.1f}s | YOLO {res['box'].get('conf', 1):.2f} | edges {len(res['points'])}"
        ax.imshow(bgr_to_rgb(overlay))
        ax.set_title(title)
        ax.axis('off')
    for ax in axes[n:]:
        ax.axis('off')
    plt.show()

plot_grid(PREVIEW_SECONDS, conf=0.10, edge_percentile=86, band=(0.0, 1.0))

## Exportar diagnosticos

Este bloque guarda overlays y CSV para un rango de tiempo. Ajusta `START_SEC`, `END_SEC`, `STEP_SEC` y los parametros de edges antes de correr.

In [ ]:
import csv

START_SEC = 80
END_SEC = 125
STEP_SEC = 1.0

export_dir = OUT_DIR / 'exports'
export_dir.mkdir(parents=True, exist_ok=True)
rows = []

for sec in np.arange(START_SEC, END_SEC + 1e-6, STEP_SEC):
    res = analyze_second(float(sec), conf=0.10, edge_percentile=86, band=(0.0, 1.0), show=False)
    if res is None:
        rows.append({'second': sec, 'frame_idx': '', 'has_roi': False, 'line_method': 'none'})
        continue
    x0, y0, x1, y1 = res['roi_px']
    line_src = res['refined'] or res['ransac']
    overlay = res['rectified'].copy()
    cv2.rectangle(overlay, (x0, y0), (x1, y1), (0, 180, 255), 2)
    row = {
        'second': sec,
        'frame_idx': res['frame_idx'],
        'has_roi': True,
        'yolo_conf': res['box'].get('conf', 1.0),
        'edge_points': len(res['points']),
        'line_method': 'projection' if res['refined'] is not None else ('ransac' if res['ransac'] is not None else 'none'),
    }
    if line_src is not None:
        xs = np.array([0, res['roi'].shape[1] - 1], dtype=np.float32)
        ys = line_src['slope'] * xs + line_src['intercept']
        cv2.line(overlay, (x0 + int(xs[0]), y0 + int(ys[0])), (x0 + int(xs[1]), y0 + int(ys[1])), (0, 255, 0), 3)
        row.update({
            'slope_roi': line_src['slope'],
            'intercept_roi': line_src['intercept'],
            'left_y_full': y0 + float(ys[0]),
            'right_y_full': y0 + float(ys[1]),
            'crm_px': line_src.get('crm_px'),
        })
    cv2.imwrite(str(export_dir / f"frame_{res['frame_idx']:06d}_{sec:07.2f}s.jpg"), overlay)
    rows.append(row)

csv_path = export_dir / 'edge_line_analysis.csv'
fieldnames = sorted({k for row in rows for k in row.keys()})
with csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print('CSV:', csv_path)
print('Overlays:', export_dir)
print('rows:', len(rows))

## Notas para iterar

Si la linea cae sobre el borde equivocado:

- Prueba restringir `EDGE_BAND`, por ejemplo `(0.4, 1.0)` o `(0.0, 0.55)`.
- Baja `EDGE_PERCENTILE` si faltan edges; subelo si hay ruido.
- Sube `RANSAC_TOL` si el borde es grueso o rugoso; bajalo si agarra varias bandas.
- Si YOLO detecta una caja muy alta, considera entrenar la box para que sea mas estricta alrededor del paquete que realmente queremos medir.

La clave es mirar `Sobel Y abs`, la mascara y los inliers: ahi se ve si el problema es de ROI, de preprocesamiento o de seleccion de linea.